In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import os

load_dotenv(override=True)

In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_BAUSE_URL = "https://api.groq.com/openai/v1/"
MODEL = "openai/gpt-oss-120b"
groq_client = OpenAI(api_key=GROQ_API_KEY, base_url=GROQ_BAUSE_URL)
SYSTEM_PROMPT = "You are a helpful assistant."

In [ ]:
def chat_gpt(message, history):
    hostory = [ {"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + hostory + [{"role": "user", "content": message}]
    stream = groq_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content or ''
            yield response

In [ ]:
gr.ChatInterface(
    fn=chat_gpt,
    textbox=gr.Textbox(label="Input"),
    chatbot=gr.Chatbot(label="Output")
).launch()

In [ ]:
# Connecting locally to Gemma3:4B
import requests

In [ ]:
response = requests.get("http://localhost:11434/v1/models")
gemma_client = OpenAI(base_url="http://localhost:11434/v1", api_key="gemma3:4B")

In [ ]:
def chat_gemma(message, history):
    history = [ {"role": h["role"], "content": h["content"]} for h in history]
    message = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    stream = gemma_client.chat.completions.create(
        model="gemma3:4B",
        messages=message,
        stream=True
    )
    
    response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content or ''
            yield response
            

In [ ]:
gr.ChatInterface(
    fn=chat_gemma,
    textbox=gr.Textbox(label="Input"),
    chatbot=gr.Chatbot(label="Output")
).launch()

In [ ]:
def stream_model(message, history, model):
    if model == "GPT":
        stream = chat_gpt(message, history)
    elif model == "Gemma":
        stream = chat_gemma(message, history)
    else:
        raise ValueError("Unknown model")

    for text in stream:
        yield [{"role": "assistant", "content": text}]

In [ ]:
message_input = gr.Textbox(label="Input")
message_output = gr.Chatbot(label="Output")

model_selector = gr.Radio(
    choices=["GPT", "Gemma"],
    label="Select Model",
    value="GPT",
)

chat = gr.ChatInterface(
    fn=stream_model,
    additional_inputs=[model_selector],
    title="LLMs",
    # type="messages",
    examples=[
        ["Explain the Transformer architecture to a layperson", "GPT"],
        ["Explain the Transformer architecture to an aspiring AI engineer", "Gemma"],
    ],
)

chat.launch()